Uvoz biblioteka

In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV

Učitavanje datoteke

In [2]:
file_path = "C:/Users/Lana/Desktop/faksic/semestar5/projektR/merged_features.csv"
data = pd.read_csv(file_path)
print(data.head())
print("Broj uzoraka po pojedinoj klasi")
print(data['event'].value_counts())

    rsp_mean  rsp_variance  rsp_skewness  rsp_kurtosis    rsp_cv   rsp_q025  \
0 -18.229200    164.229927      1.572565      5.287429 -0.703005 -37.661589   
1 -14.515604     71.103587     -0.307630     -1.128432 -0.580912 -30.155367   
2  -9.417282     45.033051     -0.446655     -1.106052 -0.712591 -22.318615   
3  -5.495158     25.097147     -0.678167     -0.776210 -0.911658 -15.808828   
4  -2.439874     11.713663     -0.595987     -0.725810 -1.402746  -9.660294   

     rsp_q25    rsp_q50    rsp_q75   rsp_q975  ...  P4_PSD32  P4_PSD33  \
0 -27.254009 -18.188616 -10.643854  19.667557  ...  0.120855  0.317489   
1 -21.427261 -13.842648  -6.741347  -2.350650  ...  0.041112  0.254432   
2 -15.172470  -7.788162  -2.937471  -1.019476  ...  0.033091  0.229946   
3  -9.027317  -4.417047  -1.269303   0.485936  ...  0.053646  0.255754   
4  -5.001319  -1.719943   0.220773   2.171125  ...  0.065412  0.234002   

   P4_PSD34  P4_PSD35  P4_PSD36  P4_PSD37  P4_PSD38  P4_PSD39  P4_PSD40  event  

Odvajanje ciljne varijable

In [3]:
y = data['event']
X = data.drop(columns=['event'])


Normalizacija

In [4]:
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)

# Kreiranje DataFrame-a za normalizirane podatke
normalized_data = pd.DataFrame(X_normalized, columns=X.columns)
print("\nPrimjer normaliziranih podataka:")
print(normalized_data.head())

normalized_data['event'] = y


Primjer normaliziranih podataka:
   rsp_mean  rsp_variance  rsp_skewness  rsp_kurtosis    rsp_cv  rsp_q025  \
0  0.089573      0.129901      0.656883      0.100556  0.191655  0.485345   
1  0.173293      0.056239      0.526534      0.010366  0.191656  0.575012   
2  0.288231      0.035618      0.516896      0.010680  0.191654  0.668628   
3  0.376653      0.019848      0.500846      0.015317  0.191652  0.746392   
4  0.445533      0.009262      0.506543      0.016026  0.191647  0.819841   

    rsp_q25   rsp_q50   rsp_q75  rsp_q975  ...  P4_PSD31  P4_PSD32  P4_PSD33  \
0  0.103436  0.098608  0.074695  0.346889  ...  0.005581  0.002995  0.006386   
1  0.235224  0.194579  0.156933  0.018531  ...  0.002864  0.000939  0.005088   
2  0.376695  0.328279  0.237093  0.038382  ...  0.002661  0.000732  0.004584   
3  0.515685  0.402723  0.272247  0.060833  ...  0.002389  0.001262  0.005115   
4  0.606744  0.462282  0.303647  0.085964  ...  0.003083  0.001566  0.004667   

   P4_PSD34  P4_PSD35 

Mapiranje klasa

In [5]:
normalized_data['event'] = normalized_data['event'].map({'A': 0, 'B': 1, 'C': 2, 'D':3})

Podjela podataka na skup za testiranje i treniranje

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    normalized_data.drop(columns=['event']),  
    normalized_data['event'],                
    test_size=0.2,                           
    random_state=42                          
)

print("\nNova raspodjela podataka:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


Nova raspodjela podataka:
X_train: (15207, 378)
y_train: (15207,)
X_test: (3802, 378)
y_test: (3802,)


Kreiranje i treniranje modela

In [7]:
rf_model = RandomForestClassifier(
    n_estimators=100,  
    random_state=42     
)

rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

Predviđanje točnosti

In [11]:
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Točnost: {accuracy:.2f}")

Točnost: 0.93


Pokušaj podizanja točnosti

In [13]:
param_grid = {
    'n_estimators': [100, 200, 300],  # Manje opcija za broj stabala
    'max_depth': [None, 10, 20],      # Smanjen broj dubina
    'min_samples_split': [2, 5],      # Manje opcija za podjelu
    'min_samples_leaf': [1, 2],       # Manje opcija za listove
    'max_features': ['sqrt', 'log2']  # Uklonjen parametar `None`
}

rf_model2 = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(estimator=rf_model2, param_grid=param_grid, 
                           cv=3, scoring='accuracy', verbose=2, n_jobs=-1)

grid_search.fit(X_train, y_train)

print("Najbolji parametri:", grid_search.best_params_)


Fitting 3 folds for each of 72 candidates, totalling 216 fits


c:\Users\Lana\AppData\Local\Programs\Python\Python39\lib\site-packages\numpy\ma\core.py:2846: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Najbolji parametri: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}


Provjeravanje točnosti s najboljim parametrimas

In [14]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Najbolja točnost: {accuracy:.2f}")

Najbolja točnost: 0.93


In [15]:
print("Izvještaj klasifikacije:")
print(classification_report(y_test, y_pred))

print("Matrica konfuzije:")
print(confusion_matrix(y_test, y_pred))

Izvještaj klasifikacije:
              precision    recall  f1-score   support

           0       0.90      0.99      0.94      2209
           1       1.00      0.11      0.19       103
           2       0.98      0.98      0.98      1319
           3       1.00      0.13      0.24       171

    accuracy                           0.93      3802
   macro avg       0.97      0.55      0.59      3802
weighted avg       0.93      0.93      0.90      3802

Matrica konfuzije:
[[2194    0   15    0]
 [  86   11    6    0]
 [  21    0 1298    0]
 [ 146    0    2   23]]
